# 04 — Features de ritmo

**Objetivo:** construir features que midan el **ritmo** cardíaco, que las 105
features actuales no capturan (notebook 03, secciones 1 y 10).
Al terminar deberías poder explicar:
- qué es la **envolvente** de energía y cómo se obtiene;
- cómo se detectan los ruidos cardíacos como "picos" de la envolvente;
- qué es la **autocorrelación** y por qué sirve para medir regularidad;
- para qué arritmias ayudan estas features, y cuáles son sus límites.

En la notebook 03 vimos que fibrilación auricular, bloqueo AV y S3 eran los
tipos peor separados de los normales. Los dos primeros son trastornos del
ritmo: el timbre de cada latido puede ser normal, lo que cambia es *cuándo*
ocurre.

In [ ]:
import sys
from pathlib import Path

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal
from sklearn.metrics import roc_auc_score

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.paths import AUDIO, DATA_DIR, OUT_DIR

SR = 4000        # frecuencia nativa
FS_ENV = 100     # la envolvente se guarda con 100 valores por segundo (uno cada 10 ms)

hs = pd.read_csv(DATA_DIR / "HS.csv")


def cargar(file_id):
    y, _ = librosa.load(AUDIO["HS"] / f"{file_id}.wav", sr=SR)
    return y


def primer_id(tipo):
    return hs.loc[hs["Heart Sound Type"] == tipo, "Heart Sound ID"].iloc[0]


# Para el normal usamos F_N_LUSB, uno de los de envolvente más limpia; en la
# sección 2 vemos que no todos son así.
EJEMPLOS = {"Normal": "F_N_LUSB",
            **{t: primer_id(t) for t in ["Atrial Fibrillation", "AV Block", "Tachycardia"]}}

## 1. La envolvente: una curva que sube con cada ruido

Para encontrar los ruidos cardíacos no necesitamos todas las oscilaciones del
sonido, solo saber **cuándo hay energía**. La envolvente se obtiene en cuatro pasos:

1. **Pasa banda 25–400 Hz:** nos quedamos con la banda cardíaca (notebook 02).
2. **Normalizar por RMS:** para que el volumen de la grabación no importe.
3. **Rectificar** (valor absoluto): la señal oscila arriba y abajo de cero;
   al tomar el valor absoluto, toda la energía queda "para arriba".
4. **Suavizar** con un pasa bajo de 20 Hz: borra las oscilaciones rápidas y
   deja una curva lenta que sube con cada ruido y baja en los silencios. 20 Hz
   conserva eventos de unos 50 ms (la duración de R1 o R2).

Después la guardamos con 100 valores por segundo, que alcanza para ver
eventos separados por decenas de milisegundos.

In [ ]:
def envolvente(y):
    """Envolvente de energía, 100 valores por segundo, estandarizada (media 0, desvío 1)."""
    y = signal.sosfiltfilt(signal.butter(4, (25, 400), "bandpass", fs=SR, output="sos"), y)
    y = y / np.sqrt(np.mean(y ** 2))
    e = signal.sosfiltfilt(signal.butter(4, 20, "lowpass", fs=SR, output="sos"), np.abs(y))
    e = e[:: SR // FS_ENV]
    return (e - e.mean()) / e.std()


y = cargar(EJEMPLOS["Normal"])
e = envolvente(y)
SEG = 4

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(np.arange(SEG * SR) / SR, y[:SEG * SR], lw=0.4)
axes[0].set(ylabel="Amplitud", title=f"{EJEMPLOS['Normal']} — sonido")
axes[1].plot(np.arange(SEG * FS_ENV) / FS_ENV, e[:SEG * FS_ENV], color="tab:red")
axes[1].set(ylabel="Envolvente", xlabel="Tiempo (s)", title="Envolvente: un 'bulto' por cada ruido cardíaco")
fig.tight_layout()

## 2. Detectar los ruidos: los picos de la envolvente

Cada bulto es un ruido cardíaco. Para encontrarlos usamos `find_peaks`, con
dos condiciones:
- **distancia mínima de 150 ms** entre picos: dos ruidos no pueden estar más
  cerca (R1 y R2 están separados por la sístole, ~300 ms);
- **prominencia mínima de 0,5:** el pico tiene que sobresalir al menos medio
  desvío respecto de lo que lo rodea, para no contar irregularidades chicas.

Ojo: en un sonido normal **cada latido da dos picos** (R1 y R2). Los
intervalos entre picos van a alternar: corto (sístole), largo (diástole),
corto, largo...

In [ ]:
def picos(e):
    """Posiciones (en índices de la envolvente) de los ruidos detectados."""
    p, _ = signal.find_peaks(e, distance=int(0.15 * FS_ENV), prominence=0.5)
    return p


SEG = 6
fig, axes = plt.subplots(len(EJEMPLOS), 1, figsize=(11, 10), sharex=True)
for ax, (tipo, file_id) in zip(axes, EJEMPLOS.items()):
    e = envolvente(cargar(file_id))
    p = picos(e)
    t = np.arange(len(e)) / FS_ENV
    ax.plot(t, e, color="tab:red", lw=1)
    ax.plot(t[p], e[p], "kv", ms=6)
    ax.set(title=f"{tipo} ({file_id})", ylabel="Envolvente", xlim=(0, SEG))
axes[-1].set_xlabel("Tiempo (s)")
fig.tight_layout()

Qué buscar:
- **Normal:** pares de picos a intervalos regulares.
- **Fibrilación auricular:** intervalos irregulares, sin patrón.
- **Bloqueo AV:** según el grado, pausas o latidos que "faltan".
- **Taquicardia:** muchos más picos por segundo.

La detección no es perfecta: a veces un soplo o un ruido agregado produce un
pico extra, o un ruido débil no llega a la prominencia mínima. **No todos los
normales se ven tan limpios como este:** probá `F_N_RC` en la sección 7 y
vas a ver una envolvente con muchos picos chicos donde cuesta reconocer los
pares R1-R2. Detectar con
precisión R1 y R2 (**segmentar** el sonido cardíaco) es un problema de
investigación en sí mismo; los métodos más usados recurren a modelos
estadísticos más complejos. Acá usamos una versión simple a propósito, para
que se entienda.

## 3. La autocorrelación: medir regularidad sin tener que acertar cada latido

La **autocorrelación** compara la envolvente consigo misma, corrida un cierto
tiempo (el *retardo*). Si el ritmo es regular con período T, al correrla
exactamente T los bultos vuelven a coincidir y la correlación es alta. Si el
ritmo es irregular, ningún corrimiento hace coincidir todo y la
autocorrelación es baja en todos los retardos.

De ahí salen dos features:
- **Regularidad:** el valor más alto de la autocorrelación entre 0,3 y 2 s de
  retardo (frecuencias de 30 a 200 lpm). Cerca de 1 = muy regular.
- **Frecuencia cardíaca estimada:** 60 / el retardo donde está ese máximo.

In [ ]:
def autocorrelacion(e):
    ac = np.correlate(e, e, mode="full")[len(e) - 1:]
    return np.arange(len(ac)) / FS_ENV, ac / ac[0]


RANGO = (0.3, 2.0)   # segundos de retardo considerados

fig, ax = plt.subplots(figsize=(10, 4))
for tipo in ["Normal", "Atrial Fibrillation"]:
    lags, ac = autocorrelacion(envolvente(cargar(EJEMPLOS[tipo])))
    ax.plot(lags, ac, label=f"{tipo} ({EJEMPLOS[tipo]})")
ax.axvspan(*RANGO, color="gray", alpha=0.15, label="rango buscado (30–200 lpm)")
ax.set(xlim=(0, 3), xlabel="Retardo (s)", ylabel="Autocorrelación",
       title="Autocorrelación de la envolvente")
ax.legend()
fig.tight_layout()

## 4. Las cinco features de ritmo

| Feature | Qué mide | Qué esperamos |
|---|---|---|
| `fc_lpm` | frecuencia cardíaca estimada por autocorrelación | alta en taquicardia |
| `regularidad` | máximo de la autocorrelación (0,3–2 s) | baja en fibrilación auricular |
| `eventos_por_s` | ruidos detectados por segundo | alta en taquicardia |
| `cv_intervalos` | coeficiente de variación (desvío / media) de los intervalos entre picos | alto si los intervalos varían mucho |
| `irregularidad` | cuánto cambia cada intervalo respecto del de **dos posiciones después** | baja si el ritmo es regular |

`irregularidad` compara cada intervalo con el que está dos lugares más
adelante porque, en un ritmo normal, los intervalos alternan
sístole-diástole: hay que comparar sístole con sístole y diástole con
diástole. En un ritmo regular ese cambio es chico; en uno irregular, grande.

In [ ]:
def features_ritmo(y):
    e = envolvente(y)
    lags, ac = autocorrelacion(e)
    en_rango = (lags >= RANGO[0]) & (lags <= RANGO[1])
    i = np.argmax(np.where(en_rango, ac, -np.inf))
    p = picos(e)
    iv = np.diff(p) / FS_ENV
    return {
        "fc_lpm": 60 / lags[i],
        "regularidad": ac[i],
        "eventos_por_s": len(p) / (len(y) / SR),
        "cv_intervalos": iv.std() / iv.mean(),
        "irregularidad": np.median(np.abs(iv[2:] - iv[:-2]) / ((iv[2:] + iv[:-2]) / 2)),
    }


ritmo = pd.DataFrame([{"file_id": r["Heart Sound ID"], "tipo": r["Heart Sound Type"],
                       **features_ritmo(cargar(r["Heart Sound ID"]))} for _, r in hs.iterrows()])
COLS = ["fc_lpm", "regularidad", "eventos_por_s", "cv_intervalos", "irregularidad"]
print(ritmo.groupby("tipo")[COLS].median().round(2).to_string())

### ¿La frecuencia cardíaca estimada es creíble?

Miremos los normales. El maniquí reproduce un ritmo normal, así que todos
deberían dar valores parecidos (60–100 lpm):

In [ ]:
print(ritmo.loc[ritmo.tipo == "Normal", ["file_id", "fc_lpm", "regularidad"]].round(2).to_string(index=False))

Algunos dan valores absurdos (por ejemplo, ~35 o ~115 lpm). Es un error típico
de la autocorrelación:
- si el máximo cae en **dos latidos** en vez de uno, la frecuencia sale a la mitad;
- si cae en el intervalo **R1→R2** (que también se repite), sale casi al doble.

Por eso `fc_lpm` es poco confiable con este método simple. `regularidad`, en
cambio, no necesita acertar cuál es el período: solo mide si *algún*
corrimiento hace coincidir la envolvente consigo misma.

## 5. ¿Ayudan donde las features espectrales fallaban?

Para cada arritmia, comparamos sus grabaciones contra las normales, igual que
en la notebook 03, sección 10. Usamos `max(AUC, 1 − AUC)`: 0,5 = no separa;
1 = separa perfecto. Como referencia ponemos la **AUC mediana de las 105
features espectrales** para ese mismo tipo.

**Advertencias:**
- Hay 3 o 4 grabaciones por arritmia contra 9 normales: un AUC puede cambiar
  mucho con una sola grabación. Es una tendencia, no un resultado.
- Los parámetros del método (20 Hz, 150 ms, prominencia 0,5) se eligieron por
  razones fisiológicas, **no** para maximizar estos números. Si los
  ajustáramos mirando esta tabla, los AUC saldrían optimistas.

In [ ]:
espectrales = pd.read_csv(OUT_DIR / "hs_features.csv")
X_esp = espectrales.drop(columns=["file_id", "label", "label_binary"])


def separacion(valores, y):
    a = roc_auc_score(y, valores)
    return max(a, 1 - a)


filas = []
for tipo in ["Atrial Fibrillation", "AV Block", "Tachycardia"]:
    m = ritmo.tipo.isin(["Normal", tipo])
    y = (ritmo.tipo[m] == tipo).astype(int)
    m_esp = espectrales.label.isin(["Normal", tipo])
    y_esp = (espectrales.label[m_esp] == tipo).astype(int)
    fila = {"tipo": tipo, "grabaciones": int(y.sum()),
            "105 espectrales (mediana)": X_esp[m_esp].apply(lambda c: separacion(c, y_esp)).median()}
    fila.update({c: separacion(ritmo.loc[m, c], y) for c in COLS})
    filas.append(fila)
print(pd.DataFrame(filas).round(2).to_string(index=False))

Preguntas para mirar la tabla:
- ¿Para qué arritmia ayudan más las features de ritmo? ¿Cuál las mide mejor?
- ¿Tiene sentido fisiológico que `eventos_por_s` funcione para la taquicardia
  y `regularidad` para la fibrilación auricular?
- ¿Qué pasa con el bloqueo AV? Pensá que hay distintos grados: algunos solo
  alargan un intervalo y no producen irregularidad.

Distribución de cada feature por tipo, para ver qué hay detrás de cada AUC:

In [ ]:
TIPOS = ["Normal", "Atrial Fibrillation", "AV Block", "Tachycardia"]
fig, axes = plt.subplots(1, len(COLS), figsize=(16, 4))
for ax, col in zip(axes, COLS):
    for i, tipo in enumerate(TIPOS):
        v = ritmo.loc[ritmo.tipo == tipo, col]
        ax.scatter(np.full(len(v), i) + np.random.uniform(-0.1, 0.1, len(v)), v, alpha=0.7)
    ax.set(xticks=range(len(TIPOS)), title=col)
    ax.set_xticklabels(["Normal", "FA", "Bloqueo AV", "Taqui"], rotation=30)
fig.tight_layout()

## 6. Guardar las features de ritmo

Las guardamos para combinarlas con las espectrales en la notebook de
comparación (11). La pregunta ahí va a ser: ¿agregar ritmo mejora la
detección de arritmias sin empeorar el resto?

In [ ]:
ritmo.to_csv(OUT_DIR / "hs_ritmo.csv", index=False)
print(f"Guardado: {OUT_DIR / 'hs_ritmo.csv'}  ({len(ritmo)} grabaciones, {len(COLS)} features)")

## 7. Tu turno

Elegí una grabación y mirá su envolvente y los picos detectados. Probá
cambiar `PROMINENCIA`: ¿qué pasa con muy baja (0,2) o muy alta (2)? Mirá un
soplo o un S3: ¿aparecen picos extra?

(Cambiar la prominencia acá es para entender el método, no para elegir el
valor que mejor separa.)

In [ ]:
FILE_ID, PROMINENCIA = "F_N_RC", 0.5

e = envolvente(cargar(FILE_ID))
p, _ = signal.find_peaks(e, distance=int(0.15 * FS_ENV), prominence=PROMINENCIA)
t = np.arange(len(e)) / FS_ENV
fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(t, e, color="tab:red", lw=1)
ax.plot(t[p], e[p], "kv", ms=6)
ax.set(title=f"{FILE_ID} — {len(p)} picos con prominencia {PROMINENCIA}", xlabel="Tiempo (s)")
fig.tight_layout()
print("Intervalos entre picos (s):", np.round(np.diff(p) / FS_ENV, 2))

## Preguntas de control

1. ¿Qué es la envolvente y cuáles son los pasos para obtenerla? ¿Por qué se
   normaliza por RMS antes?
2. ¿Por qué en un sonido normal los intervalos entre picos alternan corto y
   largo? ¿Cómo lo tiene en cuenta la feature `irregularidad`?
3. Explicá con tus palabras qué es la autocorrelación y por qué un ritmo
   regular da un valor alto.
4. ¿Por qué la frecuencia cardíaca estimada a veces sale a la mitad o al
   doble? ¿Por qué `regularidad` no tiene ese problema?
5. ¿Para qué arritmias ayudaron las features de ritmo y para cuáles no?
   ¿Tiene sentido fisiológico?
6. ¿Por qué no conviene ajustar la prominencia o el filtro mirando la tabla
   de la sección 5?